In [1]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingClassifier, GradientBoostingRegressor

class Node():
    def __init__(self, feature=None, threshold=0, left=None, right=None, value=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value

    def is_leaf_node(self):
        return self.value is not None

def train_test_split(number_of_elements, testing_size):
    training_index = [i for i in range(0, number_of_elements)]
    testing_index = []
    for i in range(1, testing_size + 1):
        choice = np.random.choice(training_index)
        training_index.remove(choice)
        testing_index.append(choice)
    return training_index, testing_index

def kfold_cross_validation(number_of_folds, data_size):
    lists = []
    for i in range(0, number_of_folds):
        lists.append([])
    for j in range(0, data_size):
        list_index = j % number_of_folds
        lists[list_index].append(j)
    return lists

In [ ]:
class Regression_Tree():
    def __init__(self, minimum_samples_split=250, max_depth=100, number_of_random_features=0, features=0, hessian=None):
        self.minimum_samples_split = minimum_samples_split
        self.max_depth = max_depth
        self.features = features
        self.number_of_random_features = number_of_random_features
        self.hessian = hessian
        self.sse_decreases = np.array([])
        self.root = None

    def choose_best_split(self, X, y):
        self.features = np.random.choice(X.shape[1], self.number_of_random_features, replace=False)
        choices = []
        initial_sse = np.sum((np.mean(y) - y)**2)
        for i in self.features:
            sorted_array = np.unique(X[:, i])
            previous_element = np.inf
            midpoints = []
            for element in sorted_array:
                if previous_element == np.inf:
                    previous_element = element
                else:
                    midpoints.append(0.5 * (previous_element + element))
                    previous_element = element
            midpoints = np.array(midpoints)
            if len(midpoints) == 0:
                continue
            sse_values = {}
            for midpoint in midpoints:
                sse = 0
                left_index = np.argwhere(X[:, i] <= midpoint).flatten()
                right_index = np.argwhere(X[:, i] > midpoint).flatten()
                left_mean = np.mean(y[left_index])
                sse += np.sum((left_mean - y[left_index])**2)
                right_mean = np.mean(y[right_index])
                sse += np.sum((right_mean - y[right_index])**2)
                sse_values.update({midpoint:sse})
            best_midpoint = min(sse_values, key=sse_values.get)
            best_sse = sse_values[best_midpoint]
            choices.append([i, best_midpoint, best_sse])
        if not choices:
            return None, None, False
        best_predictor = 0
        best_midpoint = 0
        best_sse = np.inf
        for choice in choices:
            if choice[2] < best_sse:
                best_sse = choice[2]
                best_predictor = choice[0]
                best_midpoint = choice[1]
        self.sse_decreases[best_predictor] = initial_sse - best_sse
        return best_predictor, best_midpoint, True
    
    def fit(self, X, y):
        self.sse_decreases = np.zeros((X.shape[1]))
        if self.hessian is None:
            self.hessian = np.ones(y.shape)
        self.features = X.shape[1]
        self.root = self.grow_tree(X, y, self.hessian)

    def grow_tree(self, X, y, hessian, depth=0):
        if depth >= self.max_depth or len(y) < self.minimum_samples_split:
            prediction = np.sum(y) / np.sum(hessian)
            return Node(value=prediction)
        best_predictor, best_midpoint, valid_split = self.choose_best_split(X, y)
        if not valid_split:
            prediction = np.sum(y) / np.sum(hessian)
            return Node(value=prediction)
        left_mask = X[:, best_predictor] <= best_midpoint
        right_mask = ~left_mask
        left = self.grow_tree(X[left_mask], y[left_mask], hessian[left_mask], depth+1)
        right = self.grow_tree(X[right_mask], y[right_mask], hessian[right_mask], depth+1)
        return Node(best_predictor, best_midpoint, left, right)

    def predict(self, X):
        return np.array([self.traverse_tree(x, self.root) for x in X])

    def traverse_tree(self, x, node):
        if node.is_leaf_node():
            return node.value
        if x[node.feature] <= node.threshold:
            return self.traverse_tree(x, node.left)
        else:
            return self.traverse_tree(x, node.right)

    def features_importance(self):
        return self.sse_decreases / np.sum(self.sse_decreases)
    
df = pd.read_csv(r'C:\Users\mechz\Documents\Jobs_Analysis\job_postings.csv')
y = df['salary_year_avg'].to_numpy()
X = df.drop(columns=['salary_year_avg']).to_numpy()

r2_values = []
r2_values_skl = []  
for item in kfold_cross_validation(5, y.shape[0]):
    regression_tree = Regression_Tree(max_depth=100, minimum_samples_split=250, number_of_random_features=X.shape[1])
    regression_tree.fit(np.delete(X, item, axis=0), np.delete(y, item))
    predictions = regression_tree.predict(X[item])
    r2 = 1 - (np.sum((y[item] - predictions)**2)) / (np.sum((y[item] - np.mean(y[item]))**2))
    r2_values.append(r2)
    skl_regression_tree= DecisionTreeRegressor(max_depth=100, min_samples_split=250)
    model = skl_regression_tree.fit(np.delete(X, item, axis=0), np.delete(y, item))
    predictions_skl = model.predict(X[item])
    r2_skl = 1 - (np.sum((y[item] - predictions_skl)**2)) / (np.sum((y[item] - np.mean(y[item]))**2))
    r2_values_skl.append(r2_skl)
print(f'The Average R Squared Score For A Regression Tree Is {np.mean(r2_values):.4f}.')
print(f'The Average R Squared Score For Scikit-Learn\'s Regression Tree Is {np.mean(r2_values_skl):.4f}.')

The Average R Squared Score For A Regression Tree Is 0.3238.
The Average R Squared Score For Scikit-Learn's Regression Tree Is 0.3243.
[3.19333364e-05 1.34069845e-04 1.16301394e-03 2.52941365e-05
 5.67465259e-04 6.95087476e-04 7.80755508e-04 3.65499519e-03
 2.51390654e-04 9.10497807e-03 1.68011223e-03 8.29626382e-04
 9.64765411e-04 1.98197674e-04 7.90993392e-04 1.69419967e-02
 7.41074373e-05 1.09423070e-03 4.95384479e-04 8.86790310e-04
 1.77815002e-04 3.91330723e-04 3.27171721e-04 7.82792602e-04
 8.25141496e-05 3.39234722e-05 9.52155302e-05 6.18986361e-05
 8.80088756e-05 2.71461429e-03 1.63149986e-04 3.25175361e-05
 5.06389230e-05 1.67104333e-04 1.20221997e-04 8.55750010e-04
 3.47509852e-05 1.96743396e-03 7.04641513e-04 1.09307721e-03
 1.74737948e-04 1.09727779e-01 0.00000000e+00 6.59598966e-01
 2.34236889e-02 1.84929617e-02 2.08591287e-03 4.88945581e-02
 1.38291192e-02 2.40853507e-03 7.94116337e-04 8.83995292e-04
 6.65317875e-04 2.14035275e-04 1.01633956e-03 2.41596489e-04
 7.87219260

In [58]:
class Classification_Tree():
    def __init__(self, minimum_samples_split=250, max_depth=100, number_of_random_features=0, features=0):
        self.minimum_samples_split = minimum_samples_split
        self.max_depth = max_depth
        self.features = features
        self.number_of_random_features = number_of_random_features
        self.root = None

    def choose_best_split(self, X, y):
        self.features = np.random.choice(X.shape[1], self.number_of_random_features, replace=False)
        choices = []
        for i in self.features:
            sorted_array = np.unique(X[:, i])
            previous_element = np.inf
            midpoints = []
            for element in sorted_array:
                if previous_element == np.inf:
                    previous_element = element
                else:
                    midpoints.append(0.5 * (previous_element + element))
                    previous_element = element
            midpoints = np.array(midpoints)
            if len(midpoints) == 0:
                continue
            gini_impurities = {}
            for midpoint in midpoints:
                impurity = 0
                left_index = np.argwhere(X[:, i] <= midpoint).flatten()
                right_index = np.argwhere(X[:, i] > midpoint).flatten()
                left_impurity = 1 - (np.sum(y[left_index] == 0) / len(y[left_index])) ** 2 - (np.sum(y[left_index] == 1) / len(y[left_index])) ** 2
                right_impurity = 1 - (np.sum(y[right_index] == 0) / len(y[right_index])) ** 2 - (np.sum(y[right_index] == 1) / len(y[right_index])) ** 2
                impurity = (len(y[left_index]) / len(y)) * left_impurity + (len(y[right_index]) / len(y)) * right_impurity
                gini_impurities.update({midpoint:impurity})
            best_midpoint = min(gini_impurities, key=gini_impurities.get)
            best_impurity = gini_impurities[best_midpoint]
            choices.append([i, best_midpoint, best_impurity])
        if not choices:
            return None, None, False
        best_predictor = 0
        best_midpoint = 0
        best_impurity = np.inf
        for choice in choices:
            if choice[2] < best_impurity:
                best_impurity = choice[2]
                best_predictor = choice[0]
                best_midpoint = choice[1]
        return best_predictor, best_midpoint, True
    
    def fit(self, X, y):
        self.features = X.shape[1]
        self.root = self.grow_tree(X, y)

    def grow_tree(self, X, y, depth=0):
        if depth >= self.max_depth or len(y) < self.minimum_samples_split or len(np.unique(y)) == 1:
            prediction = np.argmax(np.bincount(y))
            return Node(value=prediction)
        best_predictor, best_midpoint, valid_split = self.choose_best_split(X, y)
        if not valid_split:
            prediction = np.argmax(np.bincount(y))
            return Node(value=prediction)
        left_mask = X[:, best_predictor] <= best_midpoint
        right_mask = ~left_mask
        left = self.grow_tree(X[left_mask], y[left_mask], depth+1)
        right = self.grow_tree(X[right_mask], y[right_mask], depth+1)
        return Node(best_predictor, best_midpoint, left, right)

    def predict(self, X):
        return np.array([self.traverse_tree(x, self.root) for x in X])

    def traverse_tree(self, x, node):
        if node.is_leaf_node():
            return node.value
        if x[node.feature] <= node.threshold:
            return self.traverse_tree(x, node.left)
        else:
            return self.traverse_tree(x, node.right)
        
df = pd.read_csv(r'C:\Users\mechz\Documents\Jobs_Analysis\job_postings.csv')
y = np.where(df['salary_year_avg'] > df['salary_year_avg'].median(), 1, 0)
X = df.drop(columns=['salary_year_avg']).to_numpy()

accuracy_values = []
accuracy_values_skl = []
for item in kfold_cross_validation(5, y.shape[0]):
    total = y[item].shape[0]
    classification_tree = Classification_Tree(max_depth=100, minimum_samples_split=250, number_of_random_features=X.shape[1])
    classification_tree.fit(np.delete(X, item, axis=0), np.delete(y, item))
    predictions = classification_tree.predict(X[item])
    predictions = [1 if x > 0.5 else 0 for x in predictions]
    correct = 0
    for predicted, true in zip(predictions, y[item]):
        if predicted == true:
            correct += 1
    accuracy = correct / total
    accuracy_values.append(accuracy)
    skl_classification_tree= DecisionTreeClassifier(max_depth=100, min_samples_split=250)
    model = skl_classification_tree.fit(np.delete(X, item, axis=0), np.delete(y, item))
    predictions_skl = model.predict(X[item])
    correct_skl = 0
    for predicted, true in zip(predictions_skl, y[item]):
        if predicted == true:
            correct_skl += 1
    accuracy_skl = correct_skl / total
    accuracy_values_skl.append(accuracy_skl)
print(f'The Average Accuracy For A Classification Tree Is {np.mean(accuracy_values):.4%}.')
print(f'The Average Accuracy For Scikit-Learn\'s Classification Tree Is {np.mean(accuracy_values_skl):.4%}.')

The Average Accuracy For A Classification Tree Is 74.3686%.
The Average Accuracy For Scikit-Learn's Classification Tree Is 74.2591%.


In [59]:
class Regression_Random_Forest():
    def __init__(self, number_of_trees=100, number_of_random_features=1, max_depth=100, minimum_samples_split=25):
        self.number_of_trees = number_of_trees
        self.number_of_random_features = number_of_random_features
        self.max_depth = max_depth
        self.minimum_samples_split = minimum_samples_split
        self.trees = []
    def fit(self, X, y):
        for i in range(0, self.number_of_trees):
            indices = self.select_samples(X, y)
            regression_tree = Regression_Tree(max_depth=self.max_depth, minimum_samples_split=self.minimum_samples_split, number_of_random_features=self.number_of_random_features)
            regression_tree.fit(X[indices], y[indices])
            self.trees.append(regression_tree)
    def select_samples(self, X, y):
        return np.random.choice(y.shape[0], y.shape[0], True)
    def predict(self, X):
        predictions = []
        for tree in self.trees:
            prediction = tree.predict(X)
            predictions.append(prediction)
        predictions = np.array(predictions)
        return np.mean(predictions, axis=0)

df = pd.read_csv(r'C:\Users\mechz\Documents\Jobs_Analysis\job_postings.csv')
y = df['salary_year_avg'].to_numpy()
X = df.drop(columns=['salary_year_avg']).to_numpy()

r2_values = []
r2_values_skl = []
for item in kfold_cross_validation(5, y.shape[0]):
    regression_random_forest = Regression_Random_Forest(number_of_trees=500, number_of_random_features=np.round(0.5 * X.shape[1]).astype(np.int64))
    regression_random_forest.fit(np.delete(X, item, axis=0), np.delete(y, item))
    predictions = regression_random_forest.predict(X[item])
    r2 = 1 - (np.sum((y[item] - predictions)**2)) / (np.sum((y[item] - np.mean(y[item]))**2))
    r2_values.append(r2)
    skl_regression_random_forest = RandomForestRegressor(max_features=np.round(0.5 * X.shape[1]).astype(np.int64), n_estimators=500, min_samples_split=25, max_depth=100)
    model = skl_regression_random_forest.fit(np.delete(X, item, axis=0), np.delete(y, item))
    predictions_skl = model.predict(X[item])
    r2_skl = 1 - (np.sum((y[item] - predictions_skl)**2)) / (np.sum((y[item] - np.mean(y[item]))**2))
    r2_values_skl.append(r2_skl)
print(f'The Average R Squared Score For A Regression Random Forest Is {np.mean(r2_values):.4f}.')
print(f'The Average R Squared Score For Scikit-Learn\'s Regression Random Forest Is {np.mean(r2_values_skl):.4f}.')

The Average R Squared Score For A Regression Random Forest Is 0.4345.
The Average R Squared Score For Scikit-Learn's Regression Random Forest Is 0.4278.


In [60]:
class Classification_Random_Forest():
    def __init__(self, number_of_trees=100, number_of_random_features=1, max_depth=100, minimum_samples_split=25):
            self.number_of_trees = number_of_trees
            self.number_of_random_features = number_of_random_features
            self.max_depth = max_depth
            self.minimum_samples_split = minimum_samples_split
            self.trees = []
    def fit(self, X, y):
        for i in range(0, self.number_of_trees):
            indices = self.select_samples(X, y)
            classification_tree = Classification_Tree(max_depth=self.max_depth, minimum_samples_split=self.minimum_samples_split, number_of_random_features=self.number_of_random_features)
            classification_tree.fit(X[indices], y[indices])
            self.trees.append(classification_tree)
    def select_samples(self, X, y):
        return np.random.choice(y.shape[0], y.shape[0], True)
    def predict(self, X):
        predictions = []
        for tree in self.trees:
            prediction = tree.predict(X)
            predictions.append(prediction)
        predictions = np.array(predictions)
        return  np.mean(predictions, axis=0)

df = pd.read_csv(r'C:\Users\mechz\Documents\Jobs_Analysis\job_postings.csv')
y = np.where(df['salary_year_avg'] > df['salary_year_avg'].median(), 1, 0)
X = df.drop(columns=['salary_year_avg']).to_numpy()

accuracy_values = []
accuracy_values_skl = []
for item in kfold_cross_validation(5, y.shape[0]):
    total = y[item].shape[0]
    classification_random_forest = Classification_Random_Forest(number_of_trees=500, number_of_random_features=np.round(0.5 * X.shape[1]).astype(np.int64))
    classification_random_forest.fit(np.delete(X, item, axis=0), np.delete(y, item))
    predictions = classification_random_forest.predict(X[item])
    predictions = [1 if x > 0.5 else 0 for x in predictions]
    correct = 0
    for predicted, true in zip(predictions, y[item]):
        if predicted == true:
            correct += 1
    accuracy = correct / total
    accuracy_values.append(accuracy)
    skl_classification_random_forest = RandomForestClassifier(max_features=np.round(0.5 * X.shape[1]).astype(np.int64), n_estimators=500, min_samples_split=25, max_depth=100)
    model = skl_classification_random_forest.fit(np.delete(X, item, axis=0), np.delete(y, item))
    predictions_skl = model.predict(X[item])
    correct_skl = 0
    for predicted, true in zip(predictions_skl, y[item]):
        if predicted == true:
            correct_skl += 1
    accuracy_skl = correct_skl / total
    accuracy_values_skl.append(accuracy_skl)
print(f'The Average Accuracy For A Classification Random Forest Is {np.mean(accuracy_values):.4%}.')
print(f'The Average Accuracy For Scikit-Learn\'s Classification Random Forest Is {np.mean(accuracy_values_skl):.4%}.')

The Average Accuracy For A Classification Random Forest Is 78.3512%.
The Average Accuracy For Scikit-Learn's Classification Random Forest Is 78.1557%.


In [61]:
class Regression_Gradient_Boosting():
    def __init__(self, number_of_trees=100, max_depth=5, minimum_samples_split=25, learning_rate=0.1):
        self.number_of_trees = number_of_trees
        self.max_depth = max_depth
        self.minimum_samples_split = minimum_samples_split
        self.learning_rate = learning_rate
        self.initial_prediction = 0
        self.trees = []
    def fit(self, X, y):
        self.initial_prediction = np.mean(y)
        current_prediction = np.full(shape=y.shape, fill_value=self.initial_prediction)
        for i in range(0, self.number_of_trees):
            residuals = y - current_prediction
            regression_tree = Regression_Tree(max_depth=self.max_depth, minimum_samples_split=self.minimum_samples_split, number_of_random_features=X.shape[1])
            regression_tree.fit(X, residuals)
            current_prediction = current_prediction + self.learning_rate * regression_tree.predict(X)
            self.trees.append(regression_tree)
    def predict(self, X):
        current_prediction = np.full(shape=X.shape[0], fill_value=self.initial_prediction)
        for tree in self.trees:
            current_prediction = current_prediction + self.learning_rate * tree.predict(X)
        return current_prediction

df = pd.read_csv(r'C:\Users\mechz\Documents\Jobs_Analysis\job_postings.csv')
y = df['salary_year_avg'].to_numpy()
X = df.drop(columns=['salary_year_avg']).to_numpy()

r2_values = []
r2_values_skl = []
for item in kfold_cross_validation(5, y.shape[0]):
    regression_gradient_boosting = Regression_Gradient_Boosting(number_of_trees=750, learning_rate=0.1, max_depth=5, minimum_samples_split=25)
    regression_gradient_boosting.fit(np.delete(X, item, axis=0), np.delete(y, item))
    predictions = regression_gradient_boosting.predict(X[item])
    r2 = 1 - (np.sum((y[item] - predictions)**2)) / (np.sum((y[item] - np.mean(y[item]))**2))
    r2_values.append(r2)
    skl_regression_gradient_boosting = GradientBoostingRegressor(n_estimators=750, learning_rate=0.1, max_depth=5, min_samples_split=25)
    model = skl_regression_gradient_boosting.fit(np.delete(X, item, axis=0), np.delete(y, item))
    predictions_skl = model.predict(X[item])
    r2_skl = 1 - (np.sum((y[item] - predictions_skl)**2)) / (np.sum((y[item] - np.mean(y[item]))**2))
    r2_values_skl.append(r2_skl)
print(f'The Average R Squared Score For A Regression Gradient Boosting Model Is {np.mean(r2_values):.4f}.')
print(f'The Average R Squared Score For Scikit-Learn\'s Regression Gradient Boosting Model Is {np.mean(r2_values_skl):.4f}.')

The Average R Squared Score For A Regression Gradient Boosting Model Is 0.4173.
The Average R Squared Score For Scikit-Learn's Regression Gradient Boosting Model Is 0.4200.


In [62]:
class Classification_Gradient_Boosting():
    def __init__(self, number_of_trees=100, max_depth=5, minimum_samples_split=25, learning_rate=0.1):
        self.number_of_trees = number_of_trees
        self.max_depth = max_depth
        self.minimum_samples_split = minimum_samples_split
        self.learning_rate = learning_rate
        self.initial_prediction = 0
        self.trees = []
    def fit(self, X, y):
        self.initial_prediction = np.log(np.sum(y == 1) / np.sum(y == 0))
        current_prediction = np.full(shape=y.shape, fill_value=self.initial_prediction)
        for i in range(0, self.number_of_trees):
            p = 1 / (1 + np.exp(-current_prediction))
            residuals = y - p
            regression_tree = Regression_Tree(max_depth=self.max_depth, minimum_samples_split=self.minimum_samples_split, number_of_random_features=X.shape[1], hessian=p*(1-p))
            regression_tree.fit(X, residuals)
            current_prediction = current_prediction + self.learning_rate * regression_tree.predict(X)
            self.trees.append(regression_tree)
    def predict(self, X):
        current_prediction = np.full(shape=X.shape[0], fill_value=self.initial_prediction)
        for tree in self.trees:
            current_prediction = current_prediction + self.learning_rate * tree.predict(X)
        return 1 / (1 + np.exp(-current_prediction))

df = pd.read_csv(r'C:\Users\mechz\Documents\Jobs_Analysis\job_postings.csv')
y = np.where(df['salary_year_avg'] > df['salary_year_avg'].median(), 1, 0)
X = df.drop(columns=['salary_year_avg']).to_numpy()

accuracy_values = []
accuracy_values_skl = []
for item in kfold_cross_validation(5, y.shape[0]):
    total = y[item].shape[0]
    classification_gradient_boosting = Classification_Gradient_Boosting(number_of_trees=750, learning_rate=0.1, max_depth=5, minimum_samples_split=25)
    classification_gradient_boosting.fit(np.delete(X, item, axis=0), np.delete(y, item))
    predictions = classification_gradient_boosting.predict(X[item])
    predictions = [1 if x > 0.5 else 0 for x in predictions]
    correct = 0
    for predicted, true in zip(predictions, y[item]):
        if predicted == true:
            correct += 1
    accuracy = correct / total
    accuracy_values.append(accuracy)
    skl_classification_gradient_boosting = GradientBoostingClassifier(n_estimators=750, learning_rate=0.1, max_depth=5, min_samples_split=25)
    model = skl_classification_gradient_boosting.fit(np.delete(X, item, axis=0), np.delete(y, item))
    predictions_skl = model.predict(X[item])
    correct_skl = 0
    for predicted, true in zip(predictions_skl, y[item]):
        if predicted == true:
            correct_skl += 1
    accuracy_skl = correct_skl / total
    accuracy_values_skl.append(accuracy_skl)
print(f'The Average Accuracy For A Classification Gradient Boosting Model Is {np.mean(accuracy_values):.4%}.')
print(f'The Average Accuracy For Scikit-Learn\'s Classification Gradient Boosting Model Is {np.mean(accuracy_values_skl):.4%}.')

The Average Accuracy For A Classification Gradient Boosting Model Is 77.6188%.
The Average Accuracy For Scikit-Learn's Classification Gradient Boosting Model Is 77.6084%.
